# Практична робота №5
**Тема:** Мовні моделі

## Мета роботи
Закріпити знання про використання великих мовних моделей для генерації текстів.

Завдання:

1.	Створити програму, яка з використанням моделі BERT автоматично передбачає пропущені слова в тексті з маркером [MASK].
2. Створити програму, яка буде генерувати текст (абзац).




**Крок 1.** Імпортуйте необхідні бібліотеки

*transformers (Hugging Face)

Це головна бібліотека для роботи з мовними моделями (BERT, GPT, mBERT, BLOOM, тощо).

Дозволяє завантажувати готові моделі, токенізатори та запускати їх для генерації тексту, класифікації, заповнення пропусків.

Має зручний інтерфейс pipeline, де можна швидко створити генератор тексту або заповнювач [MASK].

Pipeline — це термін, який має кілька значень: у технічному сенсі це система труб для транспортування рідин чи газів, а в переносному — послідовність етапів або процесів, які виконуються один за одним.

*torch (PyTorch)

Виконує роль "двигуна" для моделі: обчислення, робота з тензорами, використання GPU.

Без нього більшість моделей Hugging Face не працюватимуть.

Якщо є GPU, PyTorch значно пришвидшує генерацію.

*sentencepiece / tokenizers

Використовуються для роботи з багатомовними моделями (наприклад, mBERT, XLM-R).

Вони перетворюють текст на токени (частини слів), які модель розуміє.

Для української мови це особливо важливо, бо слова можуть бути довгими й змінюватися за відмінками.

*Додаткові (корисні) бібліотеки

numpy, pandas → для обробки результатів, якщо треба аналізувати великі масиви текстів.

matplotlib, seaborn → для візуалізації (наприклад, графіки ймовірностей слів).

In [3]:
# Hugging Face Transformers
from transformers import pipeline   # високорівневий інтерфейс для моделей

# PyTorch — обчислювальний бекенд
import torch

# Додатково (якщо будемо аналізувати результати чи будувати графіки)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

**Крок 2**. Вибір та реалізація моделі для української мови для генерації наступного слова.

Оскільки  завдання — передбачати пропущені слова в реченні з маркером [MASK], ми беремо BERT‑подібну модель. Для української мови добре працюють багатомовні моделі, наприклад xlm-roberta-base.

Створюємо відповідний pipeline — це готовий інструмент у бібліотеці transformers, який дозволяє швидко використовувати мовні моделі для конкретних завдань без складної настройки.

Він автоматично:

завантажує потрібну модель,

підключає токенізатор (перетворює текст у токени),

запускає модель на введеному тексті,

повертає результат у зручному форматі.

In [ ]:
# Створюємо пайплайн для завдання "fill-mask"
fill_mask = pipeline("fill-mask", model="xlm-roberta-base")

# Використовуємо його
sentence = "Нейронні мережі є <mask> складовою сучасних систем."

preds = fill_mask(sentence, top_k=5)

for i, p in enumerate(preds, 1):
    print(f"{i}. token='{p['token_str']}' | score={p['score']:.4f} | sequence={p['sequence']}")


**Завдання для самостійного виконання**
Створіть декілька інших прикладів заповнення пропуску в реченні.

Зробіть висновки.

In [ ]:
#Тут має бути Ваш код

In [ ]:
# Використовуємо його
sentence = "the cat <mask> on the mat"

preds = fill_mask(sentence, top_k=5)

for i, p in enumerate(preds, 1):
    print(f"{i}. token='{p['token_str']}' | score={p['score']:.4f} | sequence={p['sequence']}")


Тут мають бути Ваші висновки

**Крок 3.** Оцінимо якість моделі за допомогою показника Perplexity (PP) буквально означає «ступінь розгубленості» моделі.

Це міра того, наскільки добре модель передбачає наступне слово (або токен) у тексті.

Для MASK‑завдання: можна обчислити perplexity на тестовому корпусі, щоб оцінити, наскільки добре модель відновлює пропущені слова.

У Hugging Face можна обчислити perplexity через loss:

In [ ]:
from transformers import AutoModelForMaskedLM, AutoTokenizer
import torch

model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

text = "the cat sits on the mat"
inputs = tokenizer(text, return_tensors="pt")

#Ми передаємо в модель наш текст (inputs).
#кажемо моделі: «спробуй відновити саме цей текст».
#Модель порівнює свої передбачення з реальними токенами і рахує помилку (loss).

with torch.no_grad():
    outputs = model(**inputs, labels=inputs["input_ids"])
    loss = outputs.loss
    perplexity = torch.exp(loss)

print(f"Loss: {loss.item():.4f}, Perplexity: {perplexity.item():.4f}")

**Інтерпретація**
Loss = 0.0003 Це дуже маленьке значення середньої крос‑ентропії. Воно означає, що модель практично не помиляється при відновленні токенів у цьому реченні.

Perplexity = 1.0003 Перплексія близька до 1 — це ідеальний випадок.

Значення 1 означає, що модель абсолютно впевнена у своїх прогнозах (ймовірність правильних токенів ≈ 100%).

Чим більше значення (наприклад, 50, 100, 500), тим менш впевнена модель і тим гірше вона «розуміє» текст.

**Крок 4. ** Вибір і реалізація моделі для генерації українського тексту

Для української мови можна використати багатомовну модель ai-forever/mGPT

Пояснення параметрів

prompt → початок речення, яке модель буде продовжувати.

max_length → максимальна довжина тексту.

do_sample=True → дозволяє випадковість, щоб текст був більш різноманітним.

top_p → контролює, які слова враховуються (чим менше, тим більш «академічний» текст).

temperature → керує креативністю:

низьке значення (0.5) → більш передбачуваний текст,

високе (1.0+) → більш креативний і різноманітний.

num_return_sequences → кількість варіантів тексту, які модель згенерує.

In [ ]:
import torch
from transformers import pipeline, GenerationConfig

text_gen = pipeline(
    task="text-generation",
    model="ai-forever/mGPT",
    device=0 if torch.cuda.is_available() else -1
)

prompt = "Штучний інтелект активно змінює освіту в Україні, оскільки:"

gen_config = GenerationConfig(
    max_new_tokens=120,
    do_sample=True,
    top_p=0.85,
    temperature=0.7,
    num_return_sequences=1
)

out = text_gen(prompt, generation_config=gen_config)

print(out[0]["generated_text"])



**Завдання для самостійного виконання**

Поекспериментуйте з параметрами моделі.

Зробіть висновки

In [ ]:
#Тут має бути Ваш код


Тут мають бути Ваші висновки